# 🗄️ Buổi 2: Database Schema & Setup

## 🎯 Mục tiêu buổi này
✅ Chạy lại data từ Buổi 1 để đảm bảo database sẵn sàng

✅ Xem Entity Relationship Diagram (ERD) - mối quan hệ giữa bảng

✅ Phân tích data types & ý nghĩa các cột

✅ Hiểu khóa chính (Primary Key), khóa ngoại (Foreign Key)

✅ Khái niệm Fact table vs Dimension table

✅ Thực hành: 5 bài tập tạo bảng + 5 quiz ôn tập

---

# 📘 Khóa học: HR Analyst với SQL + PostgreSQL (7 buổi)

## 🎯 Tổng quan chương trình

- **Buổi 1:** 🚀 Giới thiệu dự án
- **Buổi 2:** 🗄️ Schema & Setup database
- **Buổi 3:** 👥 Employee Overview Analysis
- **Buổi 4:** 📊 Turnover Analysis
- **Buổi 5:** 💰 Salary Analysis
- **Buổi 6:** 📈 Engagement & Advanced Analysis
- **Buổi 7:** ✅ Quiz HR Analytics SQL

---

## 📋 PHẦN 1: Verify database từ Buổi 1

### 1.1 Kết nối PostgreSQL
1. Mở **pgAdmin 4** hoặc **DBeaver**
2. Kết nối database `hr_analytics`
3. Mở **Query Tool** (pgAdmin) hoặc **SQL Editor** (DBeaver)

### 1.2 Chạy verify query
```sql
-- Kiểm tra tất cả bảng đã tạo
SELECT table_name 
FROM information_schema.tables 
WHERE table_schema = 'public' 
ORDER BY table_name;
```
**Kỳ vọng kết quả:**

![img2-1.png](images/images_buoi2/img2-1.png)

### 1.3 Đếm số records mỗi bảng
```sql
SELECT 
  (SELECT COUNT(*) FROM departments) AS departments,
  (SELECT COUNT(*) FROM job_titles) AS job_titles,
  (SELECT COUNT(*) FROM employees) AS employees,
  (SELECT COUNT(*) FROM salaries) AS salaries,
  (SELECT COUNT(*) FROM performance_reviews) AS performance_reviews,
  (SELECT COUNT(*) FROM engagement_surveys) AS engagement_surveys,
  (SELECT COUNT(*) FROM leavers) AS leavers;
```
**Kỳ vọng:**

![img2-2.png](images/images_buoi2/img2-2.png)

---


## 🔍 PHẦN 2: Xem Entity Relationship Diagram (ERD)

### 2.1 Trong pgAdmin (Windows/Mac)
1. Tìm database `hr_analytics` ở panel trái -> `Schemas` -> `public`
2. Chuột phải → **ERD For Schema**
3. Một cửa sổ mới hiện lên, vẽ sơ đồ quan hệ các bảng

### 2.2 Trong DBeaver (Windows/Mac)
1. Chuột phải database `hr_analytics`
2. **ER Diagram** → **New Diagram**
3. Chuột phải trên Diagram → **Add Tables** → chọn tất cả 7 bảng

### 2.3 ERD Diagram 

**Nhận xét:** Mọi bảng khác đều **JOIN** về `employees` (trung tâm)

![img2-3.png](images/images_buoi2/img2-3.png)


## 📊 PHẦN 3: Phân tích DATA TYPES & Cột

### 3.1 Xem schema chi tiết
```sql
-- Liệt kê tất cả cột & data type -- Copy đoạn lệnh dưới đây dán vào Postgres và chạy
SELECT 
  table_name,
  column_name,
  data_type,
  is_nullable
FROM information_schema.columns
WHERE table_schema = 'public'
ORDER BY table_name, ordinal_position;

--Kỳ vọng kết quả sau khi chạy:
```
![img2-4.png](images/images_buoi2/img2-4.png)



### 3.2 Bảng `departments`
| Cột | Data Type | Nullable | Ý nghĩa |
|---|---|---|---|
| dept_id | SERIAL | ❌ | ID phòng ban (khóa chính) |
| dept_name | TEXT | ❌ | Tên phòng (Engineering, Product...) |
| location | TEXT | ❌ | Địa điểm (Hanoi, HCMC...) |
| head_id | INT | ✅ | Quản lý phòng (null = chưa gán) |

### 3.3 Bảng `employees`
| Cột | Data Type | Nullable | Ý nghĩa |
|---|---|---|---|
| emp_id | SERIAL | ❌ | ID nhân viên (PK) |
| first_name | TEXT | ❌ | Tên |
| last_name | TEXT | ❌ | Họ |
| gender | CHAR(1) | ❌ | M/F/O |
| date_of_birth | DATE | ❌ | Ngày sinh (tính tuổi) |
| hired_date | DATE | ❌ | Ngày được tuyển (tính tenure) |
| dept_id | INT | ✅ | FK → departments |
| title_id | INT | ✅ | FK → job_titles |
| manager_id | INT | ✅ | FK → employees (self-join) |
| employment_status | TEXT | ❌ | active/on_leave/terminated |
| city | TEXT | ❌ | Hanoi/HCMC/Da Nang |
| remote_status | TEXT | ❌ | Remote/Hybrid/Onsite |
| education_level | TEXT | ✅ | Bachelor/Master/PhD |

### 3.4 Bảng `salaries`
| Cột | Data Type | Nullable | Ý nghĩa |
|---|---|---|---|
| salary_id | SERIAL | ❌ | ID record (PK) |
| emp_id | INT | ✅ | FK → employees |
| base_salary | NUMERIC(12,2) | ❌ | Lương cơ bản (VND) |
| bonus | NUMERIC(12,2) | ✅ | Thưởng hàng năm |
| effective_date | DATE | ❌ | Ngày hiệu lực |

### 3.5 Bảng `performance_reviews`
| Cột | Data Type | Nullable | Ý nghĩa |
|---|---|---|---|
| review_id | SERIAL | ❌ | ID review (PK) |
| emp_id | INT | ✅ | FK → employees |
| review_date | DATE | ❌ | Ngày đánh giá |
| performance_score | INT | ❌ | 1-10 (1 tệ, 10 xuất sắc) |
| rating | TEXT | ❌ | Exceeds/Meets/Below expectation |
| promo_recommendation | BOOLEAN | ✅ | Có nên thăng chức? |

### 3.6 Bảng `engagement_surveys`
| Cột | Data Type | Nullable | Ý nghĩa |
|---|---|---|---|
| response_id | SERIAL | ❌ | ID response (PK) |
| emp_id | INT | ✅ | FK → employees |
| survey_date | DATE | ❌ | Ngày khảo sát |
| engagement_score | INT | ❌ | 1-10 (gắn bó công ty) |
| wellbeing_score | INT | ❌ | 1-10 (sức khỏe tinh thần) |
| nps_score | INT | ❌ | -100 đến 100 (NPS: sẽ giới thiệu?) |

### 3.7 Bảng `leavers`
| Cột | Data Type | Nullable | Ý nghĩa |
|---|---|---|---|
| leaver_id | SERIAL | ❌ | ID (PK) |
| emp_id | INT | ✅ | FK → employees (terminated) |
| leaving_date | DATE | ❌ | Ngày nghỉ việc |
| reason | TEXT | ✅ | Lý do nghỉ |
| exit_type | TEXT | ❌ | Voluntary/Involuntary/Retirement |
| rehired | BOOLEAN | ❌ | Có được tuyển lại? |

## 🔐 PHẦN 4: Khóa chính (Primary Key) & Khóa ngoại (Foreign Key)

### 4.1 Primary Key (PK) - Khóa chính
**Định nghĩa:** Cột duy nhất xác định 1 record, không NULL, không trùng lặp.

| Bảng | PK |
|---|---|
| departments | dept_id |
| job_titles | title_id |
| employees | emp_id |
| salaries | salary_id |
| performance_reviews | review_id |
| engagement_surveys | response_id |
| leavers | leaver_id |

### 4.2 Foreign Key (FK) - Khóa ngoại
**Định nghĩa:** Cột tham chiếu đến PK của bảng khác, đảm bảo tính nhất quán (referential integrity).

| Bảng | FK | Tham chiếu |
|---|---|---|
| employees | dept_id | departments.dept_id |
| employees | title_id | job_titles.title_id |
| employees | manager_id | employees.emp_id (self-join) |
| salaries | emp_id | employees.emp_id |
| performance_reviews | emp_id | employees.emp_id |
| engagement_surveys | emp_id | employees.emp_id |
| leavers | emp_id | employees.emp_id |

### 4.3 Ví dụ FK constraint
```sql
-- Nếu thêm employees với dept_id không tồn tại → LỖI!
INSERT INTO employees (..., dept_id, ...) 
VALUES (..., 999, ...);  -- dept_id 999 không tồn tại
-- ERROR: insert or update on table "employees" violates foreign key constraint
```
![img2-5.png](images/images_buoi2/img2-5.png)

### 4.4 Query để xem FK
```sql
SELECT
    rc.constraint_name,
    kcu.table_name AS source_table,
    kcu.column_name AS source_column,
    ccu.table_name AS target_table,
    ccu.column_name AS target_column
FROM information_schema.referential_constraints rc
JOIN information_schema.key_column_usage kcu
    ON rc.constraint_name = kcu.constraint_name
    AND rc.constraint_schema = kcu.constraint_schema
JOIN information_schema.constraint_column_usage ccu
    ON rc.constraint_name = ccu.constraint_name
    AND rc.constraint_schema = ccu.constraint_schema
WHERE rc.constraint_schema = 'public';

-- Kỳ vọng kết quả chạy được:
```

![img2-6.png](images/images_buoi2/img2-6.png)

----

## 📐 PHẦN 5: Khái niệm Fact Table vs Dimension Table

### 5.1 Dimension Table (bảng chiều)
**Mục đích:** Lưu trữ thông tin tĩnh, mô tả (lookup data)

**Đặc điểm:**
- Dữ liệu thay đổi chậm (slowly changing)
- Ít rows (hàng chục, hàng trăm)
- Giúp "mô tả" dữ liệu chính

**Ví dụ trong project:**
- `departments` - 5 rows (mô tả phòng ban)
- `job_titles` - 20 rows (mô tả vị trí)
- `employees` - 100 rows (mô tả nhân sự)

### 5.2 Fact Table (bảng sự kiện)
**Mục đích:** Lưu trữ các "facts" định lượng, sự kiện xảy ra

**Đặc điểm:**
- Dữ liệu thay đổi thường xuyên
- Nhiều rows (hàng nghìn, triệu)
- Chứa FK tham chiếu đến Dimension tables
- Chứa các số liệu định lượng (measures)

**Ví dụ trong project:**
- `salaries` - 100+ rows (lương hàng tháng/năm)
- `performance_reviews` - 100+ rows (đánh giá)
- `engagement_surveys` - 100+ rows (khảo sát)
- `leavers` - 6 rows (sự kiện nghỉ việc)

---

## 🎯 PHẦN 6: Mối quan hệ giữa các bảng (Relationships)

### 6.1 One-to-Many (1:N)
```sql
-- 1 department có thể có nhiều employees
-- Nhưng 1 employee chỉ thuộc về 1 department
SELECT 
  d.dept_name,
  COUNT(e.emp_id) as emp_count
FROM departments d
LEFT JOIN employees e ON d.dept_id = e.dept_id
GROUP BY d.dept_name;
```
**Kỳ vọng bảng kết quả**:

![img2-7](images/images_buoi2/img2-7.png)


### 6.2 Self-Join (tham chiếu chính nó)
```sql
-- Mỗi employee có manager_id tham chiếu đến emp_id của employee khác
-- Thường dùng cho:
-- Employee – Manager
-- Parent – Child
-- Org hierarchy
SELECT 
  e.first_name || ' ' || e.last_name AS employee,
  m.first_name || ' ' || m.last_name AS manager
FROM employees e
LEFT JOIN employees m ON e.manager_id = m.emp_id
LIMIT 10;
```
**Kỳ vọng bảng kết quả**:

![img2-8](images/images_buoi2/img2-8.png)


---


## 💪 PHẦN 7: Thực hành - 3 bài tập tạo bảng (Practice Exercises)

### Bài 1: Tạo bảng `company_projects`
**Yêu cầu:**
- project_id (SERIAL, PK)
- project_name (TEXT, NOT NULL)
- dept_id (INT, FK → departments)
- budget (NUMERIC(12,2))
- start_date (DATE)
- end_date (DATE)
- status (TEXT, CHECK IN ('Active','Completed','On Hold'))

```sql
CREATE TABLE company_projects (
  project_id SERIAL PRIMARY KEY,
  project_name TEXT NOT NULL,
  dept_id INT REFERENCES departments(dept_id),
  budget NUMERIC(12,2),
  start_date DATE,
  end_date DATE,
  status TEXT CHECK(status IN ('Active','Completed','On Hold'))
);
```

**Giải thích:** Bảng dự án dùng FK kết nối `departments`, thay vì quản lý ở bảng `employees`.

---

### Bài 2: Tạo bảng `training_courses`
**Yêu cầu:**
- course_id (SERIAL, PK)
- course_name (TEXT, NOT NULL)
- provider (TEXT)
- duration_days (INT)
- cost (NUMERIC(10,2))
- level (TEXT)

```sql
CREATE TABLE training_courses (
  course_id SERIAL PRIMARY KEY,
  course_name TEXT NOT NULL,
  provider TEXT,
  duration_days INT,
  cost NUMERIC(10,2),
  level TEXT
);
```

**Giải thích:** Lưu thông tin khóa học để dùng trong bảng liên kết `employee_training`.

---

### Bài 3: Tạo bảng `employee_training` (quan hệ N:N)
**Yêu cầu:**
- enrollment_id (SERIAL, PK)
- emp_id (INT, FK → employees)
- course_id (INT, FK → training_courses)
- enrollment_date (DATE)
- completion_date (DATE)
- score (INT, CHECK 0-100)

```sql
CREATE TABLE employee_training (
  enrollment_id SERIAL PRIMARY KEY,
  emp_id INT REFERENCES employees(emp_id),
  course_id INT REFERENCES training_courses(course_id),
  enrollment_date DATE,
  completion_date DATE,
  score INT CHECK(score BETWEEN 0 AND 100)
);
```

**Giải thích:** Lưu cập nhật nhân viên tham gia khoá học, ghi điểm hoàn thành, mối quan hệ N:N.

---

## 📝 PHẦN 8: Chèn dữ liệu vào bảng vừa tạo (INSERT)

### Bài 4: Insert 5 dòng vào `company_projects`
```sql
INSERT INTO company_projects (project_name, dept_id, budget, start_date, end_date, status) VALUES
('Project Alpha', 1, 500000000, '2024-01-01', '2024-06-30', 'Active'),
('Project Beta', 2, 250000000, '2024-03-01', '2024-12-31', 'On Hold'),
('Project Gamma', 3, 300000000, '2024-02-15', '2024-09-30', 'Active'),
('Project Delta', 4, 150000000, '2024-04-01', '2024-10-31', 'Completed'),
('Project Epsilon', 5, 400000000, '2024-05-01', '2024-11-30', 'Active');
```
**Giải thích:** Đưa 5 dự án mẫu vào bảng, dùng để phân tích dự án theo phòng ban.

**Kỳ vọng bảng kết quả**:

![img2-9.png](images/images_buoi2/img2-9.png)

---

### Bài 5: Insert 5 dòng vào `training_courses`
```sql
INSERT INTO training_courses (course_name, provider, duration_days, cost, level) VALUES
('SQL for HR', 'SkillHub', 5, 4000000, 'Beginner'),
('Data Visualization', 'DataCamp', 7, 6000000, 'Intermediate'),
('People Analytics', 'Coursera', 6, 5000000, 'Intermediate'),
('Leadership Skills', 'Udemy', 4, 3500000, 'Beginner'),
('Advanced Excel', 'LinkedIn Learning', 3, 3000000, 'Beginner');
```
**Giải thích:** Thêm 5 khóa đào tạo để liên kết với nhân viên.

**Kỳ vọng bảng kết quả**:

![img2-10.png](images/images_buoi2/img2-10.png)

---

### Bải 6: Insert 5 dòng vào `employee_training`
```sql
INSERT INTO employee_training (emp_id, course_id, enrollment_date, completion_date, score) VALUES
(1, 1, '2024-01-15', '2024-01-20', 90),
(2, 1, '2024-01-15', '2024-01-20', 85),
(3, 2, '2024-01-20', '2024-01-27', 72),
(4, 3, '2024-01-25', '2024-01-31', 88),
(5, 5, '2024-02-01', '2024-02-04', 95);
```
**Giải thích:** Ghi nhận 5 lượt nhân viên tham gia khóa học.

**Kỳ vọng bảng kết quả**:

![img2-11.png](images/images_buoi2/img2-11.png)

---

## 🔧 PHẦN 9: Cập nhật và xóa dữ liệu (UPDATE + DELETE)

### 9.1 UPDATE: Điều chỉnh status của dự án
*Tại project_name = 'Project Delta' -> state: Completed*
```sql
UPDATE company_projects
SET status = 'Completed'
WHERE project_name = 'Project Delta';
```
**Giải thích:** Cập nhật trạng thái dự án đã hoàn thành.

![img2-12.png](images/images_buoi2/img2-12.png)

### 9.2 UPDATE: Chỉnh sửa điểm nhân viên
*Tại emp_id = 2 và course_id = 1 -> score = 92*
```sql
UPDATE employee_training
SET score = 92
WHERE emp_id = 2 AND course_id = 1;
```
**Giải thích:** Chỉnh điểm hoàn thành khóa học của nhân viên.

![img2-13.png](images/images_buoi2/img2-13.png)

### 9.3 DELETE: Xóa một bản ghi (mẫu)
*Tại emp_id = 5 và course_id = 5 -> delete*
```sql
DELETE FROM employee_training
WHERE emp_id = 5 AND course_id = 5;
```
**Giải thích:** Xóa 1 dòng training test; 

![img2-14.png](images/images_buoi2/img2-14.png)
---

## 📚 PHẦN 10: Tài liệu tham khảo & Links

### 10.1 PostgreSQL Documentation
- 📖 [PostgreSQL Official Docs - Data Types](https://www.postgresql.org/docs/current/datatype.html)
- 📖 [PostgreSQL - Constraints](https://www.postgresql.org/docs/current/ddl-constraints.html)
- 📖 [PostgreSQL - DDL (CREATE TABLE)](https://www.postgresql.org/docs/current/sql-createtable.html)

### 10.2 Data Analytics & HR
- 📘 [People Analytics Foundation](https://www.linkedin.com/learning/topics/people-analytics)
- 📗 [HR Metrics for Business](https://hbr.org/)
- 📙 [HR Dashboard Examples](https://tableau.com/)

### 10.3 Tools & Visualizations
- 🛠️ [dbdiagram.io](https://dbdiagram.io)
- 🛠️ [DBeaver](https://dbeaver.io)
- 🛠️ [pgAdmin 4](https://www.pgadmin.org/)

### 9.1 PostgreSQL Documentation
- 📖 [PostgreSQL Official Docs - Data Types](https://www.postgresql.org/docs/current/datatype.html)
- 📖 [PostgreSQL - Constraints](https://www.postgresql.org/docs/current/ddl-constraints.html)
- 📖 [PostgreSQL - DDL (CREATE TABLE)](https://www.postgresql.org/docs/current/sql-createtable.html)

### 9.2 Database Design & Concepts
- 📘 [Star Schema & Dimensional Modeling](https://en.wikipedia.org/wiki/Star_schema)
- 📗 [Normalization - Wikipedia](https://en.wikipedia.org/wiki/Database_normalization)
- 📙 [ERD - Entity Relationship Diagram](https://www.lucidchart.com/pages/er-diagrams)
- 📕 [Primary & Foreign Keys - Oracle Docs](https://docs.oracle.com/en/database/oracle/oracle-database/21/sqlrf/constraint.html)

### 9.3 Best Practices
- 🎯 [Database Naming Conventions](https://www.sqlshack.com/en/database-naming-conventions/)
- 🎯 [When to use SERIAL vs UUID](https://www.postgresql.org/docs/current/datatype-uuid.html)
- 🎯 [Data Types Best Practices](https://www.cybertec-postgresql.com/en/postgresql-data-types/)

### 9.4 Tools & Visualization
- 🛠️ [dbdiagram.io](https://dbdiagram.io) - Vẽ ERD online
- 🛠️ [DBeaver - Free SQL Client](https://dbeaver.io)
- 🛠️ [pgAdmin 4](https://www.pgadmin.org/)
- 🛠️ [Datagrip - JetBrains IDE](https://www.jetbrains.com/datagrip/)

### 9.5 Bài viết hữu ích (Tiếng Việt)
- 📄 [PostgreSQL Cơ bản - TechBlog](https://techblog.vn/)
- 📄 [Database Design - Viblo](https://viblo.asia/)
- 📄 [SQL Optimization - Medium](https://medium.com/)

## 🎓 Kết luận Buổi 2

✅ **Bạn vừa hoàn thành:**
- Verify database từ Buổi 1
- Xem ERD - hiểu mối quan hệ giữa 7 bảng
- Phân tích 7 data types chính (SERIAL, TEXT, DATE, INT, NUMERIC, BOOLEAN, CHAR)
- Nắm concept khóa chính (PK) & khóa ngoại (FK)
- Phân biệt Fact table vs Dimension table
- Thực hành 8 bài tập CREATE TABLE, INSERT INTO & UPDATE - DELETE
- Ôn tập 5 quiz từ cơ bản → nâng cao

📅 **Buổi 3**: Employee Overview Analysis - bắt đầu viết query
- SELECT, WHERE, ORDER BY cơ bản
- JOIN các bảng
- Aggregate functions (COUNT, SUM, AVG)

---

💡 **Ghi chú quan trọng:**
- PostgreSQL là Open Source, miễn phí, mạnh mẽ.
- ERD giúp bạn "nhìn thấy" dữ liệu trước khi viết query.
- PK + FK đảm bảo tính nhất quán dữ liệu (Data Integrity).
- Fact/Dimension schema giúp tối ưu truy vấn phân tích.

🚀 **Hãy thực hành 5 bài tập trên để nắm vững hơn!**